# 3. Матрица, правые части и бесконечный свободный хвост

Это основной вычислительный практикум. Мы печатаем все элементы маленькой
матрицы, решаем её общим методом и сопоставляем с специализированной прогонкой.

In [ ]:
from pathlib import Path
import sys, time, platform
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebook can be started from the repo root or notebooks/course.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/lighthit').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the LightHit repository')
sys.path.insert(0, str(ROOT / 'src'))
from lighthit import Medium, SolverSettings, PointGreenSolver

# Explicit public test medium. No private provider is imported.
medium = Medium(0.04, 0.05, 0.7, 1.35, 450.0, 'course-synthetic')
np.set_printoptions(precision=7, suppress=True)
print('Python:', sys.executable)
print('Platform:', platform.platform())

## 3.1. Каждая степень задаёт строку

Разложим $h=\sum h_\ell p_\ell$. Рекурсия
$\mu p_\ell=a_\ell p_{\ell-1}+a_{\ell+1}p_{\ell+1}$,
$a_\ell=\ell/\sqrt{4\ell^2-1}$, $a_0=0$, даёт
$$ik a_\ell h_{\ell-1}+(d_0-\gamma_\ell)h_\ell+ik a_{\ell+1}h_{\ell+1}
=\sqrt2\,\delta_{\ell0}.$$

Первые строки:
$$
(\mu_a-i\omega/v)h_0+\frac{ik}{\sqrt3}h_1=\sqrt2,
$$
$$\frac{ik}{\sqrt3}h_0+(d_0-\gamma_1)h_1+\frac{2ik}{\sqrt{15}}h_2=0,$$
$$\frac{2ik}{\sqrt{15}}h_1+(d_0-\gamma_2)h_2+\frac{3ik}{\sqrt{35}}h_3=0.$$

В третьей строке есть $h_3$: матрица пока бесконечна. Степень $L$ задаёт
$\gamma_{\ell>L}=0$, но не зануляет $h_{\ell>L}$.

## 3.2. Исключение хвоста

Выше $L$ правая часть равна нулю. Допустимое затухающее решение свободной
рекурсии пропорционально $b_\ell$, поэтому $h_{L+1}=r_Lh_L$,
$r_L=b_{L+1}/b_L$. Конечная матрица:
$$
M_{\ell j}=(d_0-\gamma_\ell)\delta_{\ell j}
+ik a_\ell\delta_{j,\ell-1}+ik a_{\ell+1}\delta_{j,\ell+1}
+ik a_{L+1}r_L\delta_{\ell L}\delta_{jL}.
$$
Строки и столбцы $0,\ldots,L$. Последнее слагаемое — весь свободный хвост.
Матрица комплексно-симметрична, но обычно не эрмитова.

In [ ]:
from lighthit.angular import free_moments_and_tail,solve_tail_system,dense_finite_rank_reference
L=3;k=.2;omega=0.;d0=medium.extinction_per_m-1j*omega/medium.speed_m_per_ns
b,tail=free_moments_and_tail(np.array([k]),d0,L);b=b[0];rho=tail[0]
a=np.arange(1,L+2)/np.sqrt(4*np.arange(1,L+2)**2-1)
gamma=medium.scattering_per_m*medium.g**np.arange(L+1)
Mfree=np.diag(np.full(L+1,d0,dtype=complex))+np.diag(1j*k*a[:-1],1)+np.diag(1j*k*a[:-1],-1)
Mfree[-1,-1]+=1j*k*a[-1]*rho
M=Mfree-np.diag(gamma)
f=np.zeros(L+1,complex);f[0]=np.sqrt(2)
print('a =',a);print('gamma =',gamma);print('b =',b);print('r_L =',rho)
print('M_free =\n',Mfree);print('M =\n',M);print('f =',f)
h=np.linalg.solve(M,f)
print('h =',h)

## 3.3. Две правые части для первого и всех остальных порядков

Свободная компонента уже известна: $h^{(0)}=b$. Затем
$$M_{\rm free}h^{(1)}=\Gamma b,\qquad M h^{(\ge2)}=\Gamma h^{(1)},$$
$\Gamma=\operatorname{diag}(\gamma_0,\ldots,\gamma_L)$.
Число столкновений во второй системе не ограничено. Для каждой сохранённой
угловой модели суммируются все порядки $2,3,\ldots$.

In [ ]:
rhs1=gamma*b;h1=np.linalg.solve(Mfree,rhs1)
rhs2=gamma*h1;h2=np.linalg.solve(M,rhs2)
print('rhs1 =',rhs1);print('rhs2 =',rhs2)
print('h1 =',h1);print('h>=2 =',h2)
np.testing.assert_allclose(h,b+h1+h2,rtol=1e-13,atol=1e-13)
reference=dense_finite_rank_reference(k,omega,medium,L,L,quadrature_order=512)
np.testing.assert_allclose(h,reference,rtol=2e-11,atol=2e-11)
h2_fast=solve_tail_system([k],d0,tail,gamma,rhs2[None,:])[0]
np.testing.assert_allclose(h2,h2_fast,rtol=1e-13,atol=1e-13)
print('All three comparisons passed')

## 3.4. Отношения нельзя получать делением двух машинных нулей

Для ненормированных моментов $F_\ell$:
$$F_{\ell+1}=\frac{(2\ell+1)zF_\ell-\ell F_{\ell-1}}{\ell+1},\quad z=id_0/k.$$
При обратном ходе вычисляем
$$R_{\ell-1}=\frac{\ell}{(2\ell+1)z-(\ell+1)R_\ell},\qquad
r_\ell=\sqrt{\frac{2\ell+3}{2\ell+1}}R_\ell.$$
Нужна минимальная ветвь. При $\omega=0$ её затухание
$\eta=\operatorname{arsinh}(\mu_t/k)$. Малый $k$ даёт большое $\eta$:
дорогая область находится около переключения прямой/обратной рекурсии.

In [ ]:
from lighthit.angular import _free_moments_and_ratios
bb,rr=_free_moments_and_ratios([1e-5],.09,160)
print('zero moments:',np.count_nonzero(bb==0),'nonzero ratios:',np.count_nonzero(rr))
assert np.isfinite(rr).all() and np.any((bb[0,:-1]==0)&(rr[0,:-1]!=0))
Mbad=M.copy();Mbad[-1,-1]-=1j*k*a[-1]*rho
bad=np.linalg.solve(Mbad,f)
print('Relative low-moment change when discarding the tail:',abs(bad[0]/h[0]-1))

## Задания

Повторить печать для $L=2$ и $\omega\ne0$. Найти ошибку, если в последней
строке убрать хвост. Различать: степень $L$ оператора рассеяния, выходную
степень $J$ и число столкновений. Написать собственную прогонку и сравнить
не только решение, но и невязку $Mh-f$.

Код: `free_moments_and_tail`, `solve_tail_system`, `angular_components`.